# Shirt numbers: best views, enhanced, merged over frames, read automatically

For every track piece: the frames where the player is biggest (not evenly spaced), the shirt area at
full video resolution, contrast-enhanced, and the best views merged with their neighbouring frames to
remove night noise. A digits-only text reader (EasyOCR) reads each view; Claude votes per piece and
checks the reader on team B, whose numbers you already confirmed.

Output per segment in `MyDrive/Playbook/review/match_video_2/<segment>/`: `jersey_reads.csv.gz` (every
read), `numbers.jpg` + `numbers.json` (enhanced shirt photos for the app).

**Runtime:** L4 GPU, about 3-5 minutes per segment. **Runtime -> Run all**; do not edit cells.
If it disconnects, **Run all again**: finished segments are skipped.

In [ ]:
# Cell 1 — GPU, Drive, inputs
import os, json, time, glob, shutil, subprocess, sys
g = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], capture_output=True, text=True) if shutil.which('nvidia-smi') else None
if g is None or g.returncode != 0:
    raise SystemExit('No GPU. Runtime -> Change runtime type -> L4 GPU, then re-run.')
print('GPU:', g.stdout.strip())
from google.colab import drive
drive.mount('/content/drive')
PBD = '/content/drive/MyDrive/Playbook'
MATCH = 'match_video_2'
CLIPS = sorted(glob.glob(f'{PBD}/video_downloads/{MATCH}_clips/*.mp4'))
REVIEW = f'{PBD}/review/{MATCH}'
NAMES = [os.path.basename(c)[:-4] for c in CLIPS]
missing = [f'{REVIEW}/{n}/tracks_small.csv.gz' for n in NAMES if not os.path.exists(f'{REVIEW}/{n}/tracks_small.csv.gz')]
if missing or not CLIPS:
    raise SystemExit(f'Missing in Drive: {missing[:3] or "the match clips"}')
print(len(CLIPS), 'segments')

In [ ]:
# Cell 2 — Our repo and the digit reader
PB, BRANCH = '/content/playbook', 'claude/setup-gpu-video-testing-JhgUH'
if not os.path.isdir(PB + '/.git'):
    subprocess.run(['git', 'clone', '-q', '--depth', '1', '--branch', BRANCH,
                    'https://github.com/MuwafagQ/Playbook-program.git', PB], check=True)
else:
    subprocess.run(['git', 'fetch', '-q', '--depth', '1', 'origin', BRANCH], cwd=PB, check=True)
    subprocess.run(['git', 'reset', '-q', '--hard', 'FETCH_HEAD'], cwd=PB, check=True)
for m in [m for m in list(sys.modules) if m.split('.')[0] in ('vision', 'tools', 'core', 'geometry')]:
    del sys.modules[m]
os.chdir(PB); sys.path.insert(0, PB)
!pip install -q easyocr
import numpy as np, pandas as pd, cv2, easyocr
from tools.jersey import best_views, torso_crop, enhance, fuse, clean_number, sharpness
reader = easyocr.Reader(['en'], gpu=True, verbose=False)
print('ready')

In [ ]:
# Cell 3 — Best views, enhanced and merged, read by the digit reader (skips finished segments)
K, FUSE = 6, 2          # 6 best views per piece; the 2 best also merged with +-2 neighbouring frames
SW, SH = 96, 128         # one photo on the sheet
def read(img):
    out = reader.readtext(img, allowlist='0123456789', detail=1, paragraph=False, min_size=4, text_threshold=0.5)
    best = max(out, key=lambda r: r[2], default=None)
    return (clean_number(best[1]), float(best[2])) if best else (None, 0.0)
for clip, n in zip(CLIPS, NAMES):
    dst = f'{REVIEW}/{n}/jersey_reads.csv.gz'
    if os.path.exists(dst):
        continue
    t0 = time.time()
    t = pd.read_csv(f'{REVIEW}/{n}/tracks_small.csv.gz')
    t = t[t.class_id.isin((1, 2)) & (t.raw_tracker_id >= 0)]
    groups = []                                    # (piece, kind, centre frame, [frames], box per frame)
    for piece, g in t.groupby('raw_tracker_id'):
        if len(g) < 15:
            continue
        g = g.set_index('segment_frame', drop=False)
        v = best_views(g.reset_index(drop=True), k=K)
        for i, r in enumerate(v.itertuples()):
            f = int(r.segment_frame)
            frames = [f]
            if i < FUSE:
                frames = [x for x in range(f - 2, f + 3) if x in g.index]
            groups.append((int(piece), 'fused' if len(frames) > 1 else 'single', f, frames,
                           {x: tuple(g.loc[x, ['x1', 'y1', 'x2', 'y2']]) for x in frames}))
    need = sorted({x for gr in groups for x in gr[3]})
    crops = {}
    cap = cv2.VideoCapture(clip); fi = 0; want = set(need)
    by_frame = {}
    for gr in groups:
        for x in gr[3]:
            by_frame.setdefault(x, []).append(gr)
    while want:
        ok = cap.grab()
        if not ok:
            break
        if fi in want:
            ok, frame = cap.retrieve()
            for gr in by_frame[fi]:
                crops[(gr[0], gr[2], fi)] = torso_crop(frame, gr[4][fi])
            want.discard(fi)
        fi += 1
    cap.release()
    rows, sheet_imgs = [], {}
    for piece, kind, f, frames, boxes in groups:
        cs = [crops[(piece, f, x)] for x in frames if (piece, f, x) in crops]
        if not cs:
            continue
        img = enhance(fuse(cs) if kind == 'fused' else cs[0])
        num, conf = read(img)
        h = boxes[f][3] - boxes[f][1]
        rows.append((piece, f, kind, num, round(conf, 3), int(h), round(sharpness(img), 1)))
        sheet_imgs.setdefault(piece, []).append((conf if num else -1, kind == 'fused', img))
    reads = pd.DataFrame(rows, columns=['piece', 'frame', 'kind', 'number', 'conf', 'height', 'sharp'])
    # sheet: per piece its 3 best photos (most confident reads first, fused first on ties)
    pieces = sorted(sheet_imgs)
    cols = 4
    sheet = np.full((max(1, (len(pieces) + cols - 1) // cols) * SH, cols * 3 * SW, 3), 24, np.uint8)
    index = {}
    for i, p in enumerate(pieces):
        x0, y0 = (i % cols) * 3 * SW, (i // cols) * SH
        best3 = sorted(sheet_imgs[p], key=lambda z: (z[0], z[1]), reverse=True)[:3]
        for k, (_, _, img) in enumerate(best3):
            hh, ww = img.shape[:2]; s = min(SW / ww, SH / hh)
            im = cv2.resize(img, (max(1, int(ww * s)), max(1, int(hh * s))), interpolation=cv2.INTER_AREA)
            ox, oy = x0 + k * SW + (SW - im.shape[1]) // 2, y0 + (SH - im.shape[0]) // 2
            sheet[oy:oy + im.shape[0], ox:ox + im.shape[1]] = im
        index[str(p)] = [x0, y0, len(best3)]
    cv2.imwrite('/content/numbers.jpg', sheet, [cv2.IMWRITE_JPEG_QUALITY, 85])
    shutil.copy('/content/numbers.jpg', f'{REVIEW}/{n}/numbers.jpg')
    json.dump({'sheet': 'numbers.jpg', 'w': SW, 'h': SH, 'pieces': index}, open(f'{REVIEW}/{n}/numbers.json', 'w'))
    reads.to_csv('/content/jr.csv.gz', index=False, compression='gzip'); shutil.copy('/content/jr.csv.gz', dst)
    left = sum(not os.path.exists(f'{REVIEW}/{m}/jersey_reads.csv.gz') for m in NAMES)
    print(f'{n}: {reads.piece.nunique()} pieces, {len(reads)} views, {reads.number.notna().sum()} with a number, '
          f'{(time.time() - t0) / 60:.1f} min | {left} segments left')
print(f'\nDone. Tell Claude the shirt numbers are in {REVIEW}.')